<a href="https://colab.research.google.com/github/abhishekganganalli/pyspark_codes/blob/main/pyspark_retail_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructField,StructType,StringType,DataType,IntegerType
from pyspark.sql.functions import *

spark= SparkSession.builder.appName("retail_data_processing").getOrCreate()

In [5]:
# input_df = spark.read.csv('sales_data.csv', header= True, inferSchema=True)
# in prod, we should not use inferSchema=True for csv file ==> does not give good performance
sales_schema=StructType([StructField("order_id",StringType()),
                         StructField("order_date",StringType()),
                         StructField("customer_id",StringType()),
                         StructField("customer_name",StringType()),
                         StructField("product_id",StringType()),
                         StructField("product_name",StringType()),
                         StructField("category",StringType()),
                         StructField("store_id",StringType()),
                         StructField("store_name",StringType()),
                         StructField("city",StringType()),
                         StructField("state",StringType()),
                         StructField('quantity', IntegerType()),
                         StructField('unit_price', IntegerType()),
                         StructField('discount_pct', IntegerType())])

input_df = spark.read.csv('sales_data.csv', header= True, schema=sales_schema)
input_df.show()
input_df.printSchema()
input_df.count()

+----------+----------+-----------+---------------+----------+------------+-----------+--------+--------------+---------+-----------+--------+----------+------------+
|  order_id|order_date|customer_id|  customer_name|product_id|product_name|   category|store_id|    store_name|     city|      state|quantity|unit_price|discount_pct|
+----------+----------+-----------+---------------+----------+------------+-----------+--------+--------------+---------+-----------+--------+----------+------------+
|ORD0000001|2026-05-31|  CUST22129|    James Brown|      P003|  Headphones|Electronics|    S005|    Main Store|     Pune|Maharashtra|       8|      3000|           5|
|ORD0000002|2026-03-07|  CUST29205|     Mia Thomas|      P005|Office Chair|  Furniture|    S005|    Main Store|     Pune|Maharashtra|       9|      7500|           5|
|ORD0000003|2026-03-29|  CUST19584|  Daniel Martin|      P010|   Bookshelf|  Furniture|    S003|    City Store|  Chennai| Tamil Nadu|      10|      6000|           5

35547

In [10]:
# 2. Create a new column called gross_amount. --> done
# Formula: quantity × unit_price

gross_df= input_df.withColumn("gross_amount",input_df.quantity*input_df.unit_price)
gross_df.show(10,False)
gross_df.printSchema()

+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+
|order_id  |order_date|customer_id|customer_name  |product_id|product_name|category   |store_id|store_name   |city     |state      |quantity|unit_price|discount_pct|gross_amount|
+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+
|ORD0000001|2026-05-31|CUST22129  |James Brown    |P003      |Headphones  |Electronics|S005    |Main Store   |Pune     |Maharashtra|8       |3000      |5           |24000       |
|ORD0000002|2026-03-07|CUST29205  |Mia Thomas     |P005      |Office Chair|Furniture  |S005    |Main Store   |Pune     |Maharashtra|9       |7500      |5           |67500       |
|ORD0000003|2026-03-29|CUST19584  |Daniel Martin  |P010      |Bookshelf   |Furniture  |S003    |City Stor

In [11]:
# 3. Create discount_amount.
# Formula: gross_amount * discount_pct / 100
discount_df= gross_df.withColumn("discount_amount",gross_df.gross_amount*gross_df.discount_pct/100)
discount_df.show(10,False)

+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+---------------+
|order_id  |order_date|customer_id|customer_name  |product_id|product_name|category   |store_id|store_name   |city     |state      |quantity|unit_price|discount_pct|gross_amount|discount_amount|
+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+---------------+
|ORD0000001|2026-05-31|CUST22129  |James Brown    |P003      |Headphones  |Electronics|S005    |Main Store   |Pune     |Maharashtra|8       |3000      |5           |24000       |1200.0         |
|ORD0000002|2026-03-07|CUST29205  |Mia Thomas     |P005      |Office Chair|Furniture  |S005    |Main Store   |Pune     |Maharashtra|9       |7500      |5           |67500       |3375.0         |
|ORD0000003|2026-03-29|CU

In [12]:
# 4. Create a new column final_amount.
# Formula: gross_amount - discount_amount
final_df=discount_df.withColumn("final_amount",discount_df.gross_amount-discount_df.discount_amount)
final_df.show(10,False)

+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+---------------+------------+
|order_id  |order_date|customer_id|customer_name  |product_id|product_name|category   |store_id|store_name   |city     |state      |quantity|unit_price|discount_pct|gross_amount|discount_amount|final_amount|
+----------+----------+-----------+---------------+----------+------------+-----------+--------+-------------+---------+-----------+--------+----------+------------+------------+---------------+------------+
|ORD0000001|2026-05-31|CUST22129  |James Brown    |P003      |Headphones  |Electronics|S005    |Main Store   |Pune     |Maharashtra|8       |3000      |5           |24000       |1200.0         |22800.0     |
|ORD0000002|2026-03-07|CUST29205  |Mia Thomas     |P005      |Office Chair|Furniture  |S005    |Main Store   |Pune     |Maharashtra|9       |7500      |5           |675

In [13]:
# 5. Rename columns.

# customer_id     → customer_key
# product_id      → product_key
# store_id        → store_key
# unit_price      → selling_price
# discount_pct    → discount_percentage

# renamed_df = final_df.withColumnRenamed('customer_id', 'customer_key') \
#                       .withColumnRenamed('product_id', 'product_key')  \
#                       .withColumnRenamed('store_id', 'store_key') \
#                       .withColumnRenamed('unit_price', 'selling_price') \
#                       .withColumnRenamed('discount_pct', 'discount_percentage')

renamed_df= final_df.withColumnsRenamed({'customer_id': 'customer_key',
                                        'product_id': 'product_key',
                                        'store_id':'store_key',
                                        'unit_price':'selling_price',
                                        'discount_pct':'discount_percentage'})
renamed_df.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_key: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product_key: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- store_key: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- selling_price: integer (nullable = true)
 |-- discount_percentage: integer (nullable = true)
 |-- gross_amount: integer (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- final_amount: double (nullable = true)



In [14]:
# 6. Create store-level sales summary.

# For every store calculate:

# total_orders
# total_quantity
# total_gross_sales
# total_discount
# total_sales
# average_sales
# minimum_sales
# maximum_sales

store_level_summary_df= renamed_df.groupBy("store_key").agg(
    count("order_id").alias("total_orders"),
    sum("quantity").alias("total_quantity"),
    sum("discount_amount").alias("total_discount"),
    sum("final_amount").alias("total_sales"),
    avg("final_amount").alias("average_sales"),
    min("final_amount").alias("Minimum_sales"),
    max("final_amount").alias("Maximum_sales"),
)
store_level_summary_df.show()

+---------+------------+--------------+--------------+------------+-----------------+-------------+-------------+
|store_key|total_orders|total_quantity|total_discount| total_sales|    average_sales|Minimum_sales|Maximum_sales|
+---------+------------+--------------+--------------+------------+-----------------+-------------+-------------+
|     S004|        4450|         24001|    2.444284E7| 3.1543636E8|70884.57528089888|        640.0|     550000.0|
|     S001|        4397|         24670|    2.590891E7| 3.2810769E8|74620.80736866045|        640.0|     550000.0|
|     S008|        4480|         24996|    2.546857E7| 3.2411043E8|     72346.078125|        640.0|     550000.0|
|     S002|        4437|         24413|    2.474466E7| 3.2181914E8|72530.79558260085|        640.0|     550000.0|
|     S005|        4525|         25095|    2.590396E7| 3.2744504E8|72363.54475138121|        640.0|     550000.0|
|     S006|        4372|         24186|   2.5252955E7|3.28583745E8|75156.39181152791|   

In [17]:
# 7. Create category-level summary.

# For every category calculate:

# total_quantity
# total_sales
# average_sales

category_level_summary=renamed_df.groupBy("category").agg(
    sum("quantity").alias("total_quantity"),
    sum("final_amount").alias("total_sales"),
    avg("final_amount").alias("avg_sales")

)
category_level_summary.show()



+-----------+--------------+------------+------------------+
|   category|total_quantity| total_sales|         avg_sales|
+-----------+--------------+------------+------------------+
|Electronics|         97547|2.09235765E9|117660.55502446156|
|  Furniture|         59087|4.63299225E8| 43748.74645892351|
|Accessories|         39633|  4.203532E7|5859.3978254809035|
+-----------+--------------+------------+------------------+



In [26]:
# 8. Create product-level summary.

# For every product calculate:
# total_quantity
# total_sales

# Sort products by highest sales.

product_level_summary= renamed_df.groupBy("product_name").agg(
    sum("quantity").alias("total_quantity"),
    sum("final_amount").alias("total_sales")
)
product_level_summary.show()

processed_sales_df=product_level_summary.orderBy(desc("total_sales"))
processed_sales_df.show()

+------------+--------------+------------+
|product_name|total_quantity| total_sales|
+------------+--------------+------------+
|Office Chair|         19857|1.38428625E8|
|      Laptop|         19539| 9.9785675E8|
|   Bookshelf|         20053|   1.11519E8|
|       Mouse|         20132|  1.497772E7|
|Office Table|         19177|  2.133516E8|
|Mobile Phone|         18753| 4.3535125E8|
|     Printer|         19511|   2.70975E8|
|    Keyboard|         19501|   2.70576E7|
|     Monitor|         19924|  3.330396E8|
|  Headphones|         19820|  5.513505E7|
+------------+--------------+------------+

+------------+--------------+------------+
|product_name|total_quantity| total_sales|
+------------+--------------+------------+
|      Laptop|         19539| 9.9785675E8|
|Mobile Phone|         18753| 4.3535125E8|
|     Monitor|         19924|  3.330396E8|
|     Printer|         19511|   2.70975E8|
|Office Table|         19177|  2.133516E8|
|Office Chair|         19857|1.38428625E8|
|   Booksh

In [27]:
# 9. Write the processed transaction data.
# Output: processed_sales
# Format: CSV
processed_sales_df.write.mode("ignore").csv('processed_sales',header=True)

In [29]:
csv_df=spark.read.csv("sales_data.csv",header=True)
csv_df.count()

csv_df.write.parquet("parquet_data")

In [41]:
# Retail Sales ETL:-
# ----------------------

# You receive a CSV file containing approximately 50 MB of retail transaction data.

# The source contains:

# order_id
# order_date
# customer_id
# customer_name
# product_id
# product_name
# category
# store_id
# store_name
# city
# state
# quantity
# unit_price
# discount_pct

# Requirements:-
# ------------------
# 1. Read the source CSV.

# Display:

# First 10 records
# Schema
# Number of records

# 2. Create gross_amount.

# Formula:

# quantity × unit_price

# 3. Create discount_amount.

# Formula:

# gross_amount × discount_pct / 100

# 4. Create final_amount.

# Formula:

# gross_amount - discount_amount

# 5. Rename columns.

# customer_id     → customer_key
# product_id      → product_key
# store_id        → store_key
# unit_price      → selling_price
# discount_pct    → discount_percentage

# 6. Create store-level sales summary.

# For every store calculate:

# total_orders
# total_quantity
# total_gross_sales
# total_discount
# total_sales
# average_sales
# minimum_sales
# maximum_sales

# 7. Create category-level summary.

# For every category calculate:

# total_quantity
# total_sales
# average_sales

# 8. Create product-level summary.

# For every product calculate:

# total_quantity
# total_sales

# Sort products by highest sales.

# 9. Write the processed transaction data.

# Output:

# processed_sales/

# Format:

# CSV

# 10. Write the store summary.

# Output:

# store_sales_summary/

# Format:

# CSV
# Teaching Flow


#                  RETAIL CSV
#                      │
#                      ▼
#               Read DataFrame
#                      │
#                      ▼
#            ┌─────────────────┐
#            │ Transformation  │
#            └─────────────────┘
#                      │
#           ┌──────────┼──────────┐
#           ▼          ▼          ▼
#      Gross Amount  Discount   Final Amount
#           │          │          │
#           └──────────┼──────────┘
#                      ▼
#               Rename Columns
#                      │
#                      ▼
#              ┌──────────────┐
#              │   groupBy    │
#              └──────────────┘
#                      │
#           ┌──────────┼──────────┐
#           ▼          ▼          ▼
#         Store      Category    Product
#        Summary     Summary     Summary
#           │          │          │
#           └──────────┼──────────┘
#                      ▼
#                 Write Files
# =================================================



# spark write modes:-

# 1. default(error) --> if the folder does not exists, it will create it. otherwise raise an error that as folder/path already exists

# 2. overwrite --> if the folder does not exists, it will create it. Otherwise spark will overwrites the existing data in the path(old data gets deleted)

# 3. append --> if the folder does not exists, it will create it. Otherwise spark will appends/adds new data to the existing path

# 4. ignore --> if the folder does not exists, it will create it. Otherwise spark will ignore the current write


# 1. I am running a pipeline
# 		--> 1st run job success --> by mistaken my team mate reran the job  --> ignore mode

# 2. in the 1st run while writing the data, some issue happend due to which full data did not write to the path --> in 2nd run we go for overwrite

# 3. there is a spark job writing the processed data daily to a folder
# 		--> customer wants to store historical data and daily run data as well
# 		--> which mode you prefer --> Append




# Assignment:-

# 1. explain how inferschema in csv file format works? disadvantages?
# 2. explain how withColumn() func works
# 3. explain spark write modes
# 4. diff b/w withColumn() and withColumnRenamed()
# 5. diff b/w file formats:- csv, avro, parquet, ORC


from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StructField,StringType,StructType,DateType,IntegerType

sales_schema=StructType([StructField("order_id",StringType()),
                         StructField("order_date",DateType()),
                         StructField("customer_id",StringType()),
                         StructField("customer_name",StringType()),
                         StructField("product_id",StringType()),
                         StructField("product_name",StringType()),
                         StructField("category",StringType()),
                         StructField("store_id",StringType()),
                         StructField("store_name",StringType()),
                         StructField("city",StringType()),
                         StructField("state",StringType()),
                         StructField("quantity",IntegerType()),
                         StructField("unity_price",IntegerType()),
                         StructField("discount_pct",IntegerType())
                         ])

spark=SparkSession.builder.appName("Retail_clean_pipeline").getOrCreate()

input_df=spark.read.csv("sales_data.csv",header=True,schema=sales_schema)
input_df.show()
input_df.printSchema()

+----------+----------+-----------+---------------+----------+------------+-----------+--------+--------------+---------+-----------+--------+-----------+------------+
|  order_id|order_date|customer_id|  customer_name|product_id|product_name|   category|store_id|    store_name|     city|      state|quantity|unity_price|discount_pct|
+----------+----------+-----------+---------------+----------+------------+-----------+--------+--------------+---------+-----------+--------+-----------+------------+
|ORD0000001|2026-05-31|  CUST22129|    James Brown|      P003|  Headphones|Electronics|    S005|    Main Store|     Pune|Maharashtra|       8|       3000|           5|
|ORD0000002|2026-03-07|  CUST29205|     Mia Thomas|      P005|Office Chair|  Furniture|    S005|    Main Store|     Pune|Maharashtra|       9|       7500|           5|
|ORD0000003|2026-03-29|  CUST19584|  Daniel Martin|      P010|   Bookshelf|  Furniture|    S003|    City Store|  Chennai| Tamil Nadu|      10|       6000|      

In [44]:
# 2. Create gross_amount.

# Formula:

# quantity × unit_price
gross_df=input_df.withColumn("gross_amount",input_df.quantity*input_df.unity_price)
gross_df.show()

+----------+----------+-----------+---------------+----------+------------+-----------+--------+--------------+---------+-----------+--------+-----------+------------+------------+
|  order_id|order_date|customer_id|  customer_name|product_id|product_name|   category|store_id|    store_name|     city|      state|quantity|unity_price|discount_pct|gross_amount|
+----------+----------+-----------+---------------+----------+------------+-----------+--------+--------------+---------+-----------+--------+-----------+------------+------------+
|ORD0000001|2026-05-31|  CUST22129|    James Brown|      P003|  Headphones|Electronics|    S005|    Main Store|     Pune|Maharashtra|       8|       3000|           5|       24000|
|ORD0000002|2026-03-07|  CUST29205|     Mia Thomas|      P005|Office Chair|  Furniture|    S005|    Main Store|     Pune|Maharashtra|       9|       7500|           5|       67500|
|ORD0000003|2026-03-29|  CUST19584|  Daniel Martin|      P010|   Bookshelf|  Furniture|    S003

In [48]:
# 3. Create discount_amount.

# Formula:

# gross_amount × discount_pct / 100

# 4. Create final_amount.

# Formula:

# gross_amount - discount_amount

discount_df=gross_df.withColumn("discount_amount",gross_df.gross_amount*gross_df.discount_pct/100)
discount_df.show()

final_df=discount_df.withColumn("final_amount",discount_df.gross_amount-discount_df.discount_amount)
final_df.show()

+----------+----------+-----------+---------------+----------+------------+-----------+--------+--------------+---------+-----------+--------+-----------+------------+------------+---------------+
|  order_id|order_date|customer_id|  customer_name|product_id|product_name|   category|store_id|    store_name|     city|      state|quantity|unity_price|discount_pct|gross_amount|discount_amount|
+----------+----------+-----------+---------------+----------+------------+-----------+--------+--------------+---------+-----------+--------+-----------+------------+------------+---------------+
|ORD0000001|2026-05-31|  CUST22129|    James Brown|      P003|  Headphones|Electronics|    S005|    Main Store|     Pune|Maharashtra|       8|       3000|           5|       24000|         1200.0|
|ORD0000002|2026-03-07|  CUST29205|     Mia Thomas|      P005|Office Chair|  Furniture|    S005|    Main Store|     Pune|Maharashtra|       9|       7500|           5|       67500|         3375.0|
|ORD0000003|202

In [51]:
# 5. Rename columns.

# customer_id     → customer_key
# product_id      → product_key
# store_id        → store_key
# unit_price      → selling_price
# discount_pct    → discount_percentage

renamed_df=final_df.withColumnsRenamed({"customer_id":"customer_key"

})
renamed_df.show()

+----------+----------+------------+---------------+----------+------------+-----------+--------+--------------+---------+-----------+--------+-----------+------------+------------+---------------+------------+
|  order_id|order_date|customer_key|  customer_name|product_id|product_name|   category|store_id|    store_name|     city|      state|quantity|unity_price|discount_pct|gross_amount|discount_amount|final_amount|
+----------+----------+------------+---------------+----------+------------+-----------+--------+--------------+---------+-----------+--------+-----------+------------+------------+---------------+------------+
|ORD0000001|2026-05-31|   CUST22129|    James Brown|      P003|  Headphones|Electronics|    S005|    Main Store|     Pune|Maharashtra|       8|       3000|           5|       24000|         1200.0|     22800.0|
|ORD0000002|2026-03-07|   CUST29205|     Mia Thomas|      P005|Office Chair|  Furniture|    S005|    Main Store|     Pune|Maharashtra|       9|       7500| 

In [53]:
# 6. Create store-level sales summary.

# For every store calculate:

# total_orders
# total_quantity
# total_gross_sales
# total_discount
# total_sales
# average_sales
# minimum_sales

store_level_sales_summary=renamed_df.groupBy("store_id").agg(
    count("order_id").alias('total_oder')
)
store_level_sales_summary.show()

+--------+----------+
|store_id|total_oder|
+--------+----------+
|    S004|     62319|
|    S001|     62865|
|    S008|     62650|
|    S002|     62534|
|    S005|     62468|
|    S006|     61898|
|    S003|     62774|
|    S007|     62492|
+--------+----------+



In [59]:
category_level_summary=renamed_df.groupBy("category").agg(
    sum("final_amount").alias("total_sales")
)
category_level_summary.show()
sort_sales_df=category_level_summary.orderBy(desc("total_sales"))
sort_sales_df.show()





+-----------+-------------+
|   category|  total_sales|
+-----------+-------------+
|Electronics|2.96492138E10|
|  Furniture|6.520833225E9|
|Accessories|   5.875369E8|
+-----------+-------------+

+-----------+-------------+
|   category|  total_sales|
+-----------+-------------+
|Electronics|2.96492138E10|
|  Furniture|6.520833225E9|
|Accessories|   5.875369E8|
+-----------+-------------+



In [61]:
sort_sales_df.write.mode("overwrite").csv("sort_sales")
